In [0]:
from pyspark.sql.functions import col, row_number
from pyspark.sql.window import Window

# ==========================================
# 1. Setup Variables
# ==========================================
catalog = "maritime_ais"
silver_schema = "maritime_silver"
gold_schema = "maritime_gold"

print("Building the Gold Layer...")

# ==========================================
# 2. Query Silver Tables
# ==========================================
df_ais = spark.read.table(f"{catalog}.{silver_schema}.ais_locations")
df_ports = spark.read.table(f"{catalog}.{silver_schema}.port_calls")

# ==========================================
# 3. Gold Transformation (Join & Enrich)
# ==========================================
# A ship can have multiple port calls. We only want their latest/current one.
window_spec = Window.partitionBy("mmsi").orderBy(col("call_timestamp").desc())
df_latest_ports = df_ports.withColumn("rn", row_number().over(window_spec)).filter(col("rn") == 1)

# Join the latest GPS locations with their destination ports
df_gold = df_ais.join(
    df_latest_ports,
    on="mmsi",
    how="left"
).select(
    df_ais["mmsi"],
    col("latitude"),
    col("longitude"),
    col("speed_over_ground"),
    col("vessel_name"),
    col("port_name").alias("destination_port"),
    col("eta")
)

# Save to the Gold Schema (Overwrite because this is a snapshot of current state)
gold_table_name = f"{catalog}.{gold_schema}.current_vessel_status"
df_gold.write.format("delta").mode("overwrite").saveAsTable(gold_table_name)
print(f"Gold table saved to {gold_table_name}")